# Tiny M1 LoRA learning diagnostic

This notebook performs the first training experiment: 12 fixed fictional units (4 F + 8 R), 84 training questions, 84 development questions and 120 optimizer attempts. It saves a diagnostic adapter, evaluates direct/multi-hop learning and checks fresh-base reload. This is not the canonical full M1. Attach HF_TOKEN in Kaggle Secrets to preserve artifacts privately before stopping the runtime.

In [1]:
import os, subprocess, sys, tempfile
from pathlib import Path

remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect this notebook to a Kaggle GPU kernel.'
repo_dir = Path(tempfile.mkdtemp(prefix='m1-tiny-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('configs/training/m1_tiny.json').is_file(), 'Push the tiny training files first.'


Cloning into '/kaggle/working/m1-tiny-source-lmcqtwqh/repo'...


Repository commit: 5ec5960358f2a67704746dc09bf781a3c930f557


In [2]:
assert Path('/kaggle/working').is_dir(), 'Run dependency setup only in the temporary Kaggle runtime.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m1-tiny.txt'], check=True)
# This unquantized LoRA pilot does not use TorchAO. Old Kaggle builds conflict with PEFT.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); print(torch.__version__, transformers.__version__, peft.__version__); print("Unused TorchAO absent; PEFT backend check passed."); assert torch.cuda.is_available()'], check=True)


2.11.0+cu130 5.20.2 0.21.2


CompletedProcess(args=['/usr/bin/python3', '-c', 'import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__); assert torch.cuda.is_available()'], returncode=0)

In [3]:
# The token is needed now because this experiment creates an adapter to preserve.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
artifact_repo = 'tharundevulapalli/persistent-llm-unlearning-checkpoints'
api = HfApi(token=hf_token)
info = api.repo_info(artifact_repo, repo_type='model')
assert info.private, 'Use the configured private artifact repository.'
print('Private artifact storage is accessible. Token is not displayed.')


Private artifact storage is accessible. Token is not displayed.


In [4]:
import json

prep_dir = Path(tempfile.mkdtemp(prefix='m1-tiny-prepare-', dir=remote_root))
subprocess.run([sys.executable, '-m', 'src.training.tiny_m1', '--config', 'configs/training/m1_tiny.json', '--output-dir', str(prep_dir), '--mode', 'prepare'], check=True)
report = json.loads((prep_dir / 'masking_report.json').read_text())
assert report['rows'] == 84 and report['prefixes_verified'] and report['truncated_examples'] == 0
print('84 training examples prepared; answer masking verified.')


{
  "rows": 84,
  "max_tokens": 101,
  "prompt_tokens": 7153,
  "supervised_answer_tokens": 601,
  "prefixes_verified": true,
  "truncated_examples": 0
}
84 training examples prepared; answer masking verified.


In [6]:
import subprocess
import sys
from pathlib import Path

assert Path("/kaggle/working").is_dir()

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=True,
)

subprocess.run(
    [
        sys.executable, "-c",
        "from peft.import_utils import is_torchao_available; "
        "assert not is_torchao_available(); "
        "print('Backend check passed')"
    ],
    check=True,
)

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Backend check passed


CompletedProcess(args=['/usr/bin/python3', '-c', "from peft.import_utils import is_torchao_available; assert not is_torchao_available(); print('Backend check passed')"], returncode=0)

In [7]:
# This cell runs actual LoRA training on the Kaggle GPU.
output_dir = Path(tempfile.mkdtemp(prefix='m1-tiny-run-', dir=remote_root))
print('Training outputs:', output_dir, flush=True)
subprocess.run([sys.executable, '-u', '-m', 'src.training.tiny_m1', '--config', 'configs/training/m1_tiny.json', '--output-dir', str(output_dir), '--mode', 'train'], check=True)


Training outputs: /kaggle/working/m1-tiny-run-8myx_22r


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 515.29it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Tiny training attempt 10/120; answer CE=1.4398
Tiny training attempt 20/120; answer CE=0.7942
Tiny training attempt 30/120; answer CE=0.5326
Tiny training attempt 40/120; answer CE=0.3343
Tiny training attempt 50/120; answer CE=0.1798
Tiny training attempt 60/120; answer CE=0.0624
Tiny training attempt 70/120; answer CE=0.0186
Tiny training attempt 80/120; answer CE=0.0476
Tiny training attempt 90/120; answer CE=0.0323
Tiny training attempt 100/120; answer CE=0.0056
Tiny training attempt 110/120; answer CE=0.1241
Tiny training attempt 120/120; answer CE=0.0007


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 477.04it/s]


{
  "execution_complete": true,
  "pilot": {
    "passed": false,
    "checks": {
      "F/author_book": true,
      "F/multi_hop": true,
      "R/author_book": true,
      "R/multi_hop": false,
      "utility_baseline_measurable": false,
      "utility_preserved": true,
      "format": true,
      "generation_limit": true,
      "save_reload": true,
      "adapter_changed": true,
      "optimizer_budget": true
    },
    "dev": {
      "F/author_book": {
        "correct": 8,
        "count": 8,
        "accuracy": 1.0
      },
      "F/multi_hop": {
        "correct": 11,
        "count": 12,
        "accuracy": 0.9166666666666666
      },
      "R/author_book": {
        "correct": 16,
        "count": 16,
        "accuracy": 1.0
      },
      "R/multi_hop": {
        "correct": 16,
        "count": 24,
        "accuracy": 0.6666666666666666
      }
    },
    "base_utility_accuracy": 0.0,
    "post_utility_accuracy": 0.4,
    "scope": "tiny diagnostic only; not a canonical M1 or e

CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'src.training.tiny_m1', '--config', 'configs/training/m1_tiny.json', '--output-dir', '/kaggle/working/m1-tiny-run-8myx_22r', '--mode', 'train'], returncode=0)

In [8]:
summary = json.loads((output_dir / 'summary.json').read_text())
assert summary['execution_complete']
print(json.dumps({'execution_complete': summary['execution_complete'], 'pilot': summary['pilot'], 'reload_predictions_identical': summary['reload_predictions_identical'], 'optimizer_updates': summary['optimizer_updates'], 'seconds': summary['seconds']}, indent=2))
print('A failed pilot is diagnostic evidence. Preserve it rather than relabeling it as full M1.')


{
  "execution_complete": true,
  "pilot": {
    "base_utility_accuracy": 0.0,
    "checks": {
      "F/author_book": true,
      "F/multi_hop": true,
      "R/author_book": true,
      "R/multi_hop": false,
      "adapter_changed": true,
      "format": true,
      "generation_limit": true,
      "optimizer_budget": true,
      "save_reload": true,
      "utility_baseline_measurable": false,
      "utility_preserved": true
    },
    "dev": {
      "F/author_book": {
        "accuracy": 1.0,
        "correct": 8,
        "count": 8
      },
      "F/multi_hop": {
        "accuracy": 0.9166666666666666,
        "correct": 11,
        "count": 12
      },
      "R/author_book": {
        "accuracy": 1.0,
        "correct": 16,
        "count": 16
      },
      "R/multi_hop": {
        "accuracy": 0.6666666666666666,
        "correct": 16,
        "count": 24
      }
    },
    "passed": false,
    "post_utility_accuracy": 0.4,
    "scope": "tiny diagnostic only; not a canonical M1 or e

In [9]:
# Upload the adapter, optimizer snapshot, tokenizer settings and evidence privately.
import hashlib
from src.m0_smoke_test import write_json

remote_path = 'diagnostics/m1-tiny/' + output_dir.name
commit = api.upload_folder(repo_id=artifact_repo, repo_type='model', folder_path=str(output_dir), path_in_repo=remote_path, commit_message='Preserve tiny M1 diagnostic')
restore_dir = Path(tempfile.mkdtemp(prefix='m1-tiny-artifact-restore-', dir=remote_root))
for filename, expected in summary['artifact_files_sha256'].items():
    restored = hf_hub_download(repo_id=artifact_repo, filename=remote_path + '/' + filename, revision=commit.oid, token=hf_token, local_dir=restore_dir)
    assert hashlib.sha256(Path(restored).read_bytes()).hexdigest() == expected, filename
receipt = {'repo_id': artifact_repo, 'artifact_revision': commit.oid, 'path': remote_path, 'adapter_file_hashes_verified': True, 'canonical_m1': False}
write_json(output_dir / 'persistence.json', receipt)
api.upload_file(repo_id=artifact_repo, repo_type='model', path_or_fileobj=str(output_dir / 'persistence.json'), path_in_repo=remote_path + '/persistence.json', commit_message='Record verified adapter round-trip')
print(json.dumps(receipt, indent=2))


README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

diagnostics/m1-tiny/m1-tiny-run-8myx_22r(…): reconstructing file:   0%|          |  0.00B / 20.2MB            

diagnostics/m1-tiny/m1-tiny-run-8myx_22r(…): downloading bytes:           |  0.00B            

{
  "repo_id": "tharundevulapalli/persistent-llm-unlearning-checkpoints",
  "artifact_revision": "07d23d8b904ab48401f9c6676d05bb85a14d9fbd",
  "path": "diagnostics/m1-tiny/m1-tiny-run-8myx_22r",
  "adapter_file_hashes_verified": true,
  "canonical_m1": false
}


In [10]:
# Export small evidence for Git; adapter and optimizer weights remain on private HF.
import zipfile

archive = Path(str(output_dir) + '-evidence.zip')
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(output_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
print('Download from Kaggle Output:', archive)
print('Stop the runtime only after the private receipt and small evidence ZIP are preserved.')


Download from Kaggle Output: /kaggle/working/m1-tiny-run-8myx_22r-evidence.zip
Stop the runtime only after the private receipt and small evidence ZIP are preserved.
